# Lab 4: Text and Script Normalization for Indic Languages


In [ ]:
import nltk
nltk.download('indian')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import indian
import unicodedata
import re
import string
import random
from collections import Counter
import pandas as pd
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

[nltk_data] Downloading package indian to /root/nltk_data...
[nltk_data]   Package indian is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Q1. NLTK Resources for Basic Analysis of Indic Language Text

In [ ]:
sents = indian.sents('hindi.pos')
print("Total sentences available:", len(sents))

tagged_sents = indian.tagged_sents('hindi.pos')
print(tagged_sents[0])

Total sentences available: 540
[('पूर्ण', 'JJ'), ('प्रतिबंध', 'NN'), ('हटाओ', 'VFM'), (':', 'SYM'), ('इराक', 'NNP')]


In [ ]:
all_tokens = [tok for sent in sents for tok in sent]
print("Sample tokens:", all_tokens[:15])

clean_tokens = [tok.lower() for tok in all_tokens if tok not in string.punctuation]
print("Total tokens after preprocessing:", len(clean_tokens))

Sample tokens: ['पूर्ण', 'प्रतिबंध', 'हटाओ', ':', 'इराक', 'संयुक्त', 'राष्ट्र', '।', 'इराक', 'के', 'विदेश', 'मंत्री', 'ने', 'अमरीका', 'के']
Total tokens after preprocessing: 9252


In [ ]:
total_tokens = len(clean_tokens)
vocab = set(clean_tokens)
vocab_size = len(vocab)
print("Total Tokens:", total_tokens)
print("Vocabulary Size:", vocab_size)

Total Tokens: 9252
Vocabulary Size: 2178


In [ ]:
freq = Counter(clean_tokens)
top10 = freq.most_common(10)
pd.DataFrame(top10, columns=['Token', 'Frequency'])

,Token,Frequency
0,।,493
1,के,387
2,में,268
3,की,236
4,ने,232
5,है,189
6,को,163
7,से,137
8,पर,131
9,और,129


**1(f) Challenges of Tokenization and Preprocessing for Indic Languages**

- Indic scripts are agglutinative and morphologically rich, so word boundaries are harder to detect than in English.
- Presence of matras, conjunct characters, and nukta variations causes inconsistent Unicode representations of the same word.
- Absence of capitalization makes named entity and sentence boundary detection harder.
- Code-mixing with English/Romanized text in real-world corpora adds further ambiguity.
- Limited availability of large annotated corpora and tools compared to English.

## Q2. Text Normalization for Indic Language Text

In [ ]:
noisy_samples = [
    "नमस्ते   दुनिया!!",
    "आप   कैसे हैं???",
    "मुझे   हिंदी में   बात करनी है...",
    "यह  बहुत  अच्छा है!!!",
    "क्या   हाल है  भाई???",
    "MUJHE   khana  KHANA HAI",
    "school   जाना है    kal",
    "yह  bahut  accha  hai....",
    "٥٥ लोग   आए  थे  meeting में",
    "उसने कहा हाँँँ मैं आऊँगा",
    "आज  का दिन  बहुत   अच्छाा  था",
    "फिल्म   देखने  चलते  हैं  क्याा",
    "STUDENT  ने  homework  नहीं किया",
    "price  is  १००  rupees only",
    "वहाँ  पर  बहुत  भीड़  थी...",
    "क्या  तुम  आ  सकते  हो???",
    "मीटिंग   कल  सुबह  10  बजे  है",
    "hume  jana  hoga   abhi  hi",
    "यह     गलत    है!!",
    "उसका  नाम   क्याा  है"
]
len(noisy_samples)

20

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'([!?.,]){2,}', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    digit_map = str.maketrans('٠١٢٣٤٥٦٧٨٩०१२३४५६७८९', '01234567890123456789')
    text = text.translate(digit_map)
    text = re.sub(r'[\"“”]', '', text)
    tokens = text.split()
    tokens = [t.lower() if re.match(r'^[A-Za-z]+$', t) else t for t in tokens]
    return ' '.join(tokens)

normalized_samples = [normalize_text(s) for s in noisy_samples]

In [ ]:
df_norm = pd.DataFrame({'Original': noisy_samples, 'Normalized': normalized_samples})
df_norm

,Original,Normalized
0,नमस्ते दुनिया!!,नमस्ते दुनिया!
1,आप कैसे हैं???,आप कैसे हैं?
2,मुझे हिंदी में बात करनी है...,मुझे हिंदी में बात करनी है.
3,यह बहुत अच्छा है!!!,यह बहुत अच्छा है!
4,क्या हाल है भाई???,क्या हाल है भाई?
5,MUJHE khana KHANA HAI,mujhe khana khana hai
6,school जाना है kal,school जाना है kal
7,yह bahut accha hai....,yह bahut accha hai.
8,٥٥ लोग आए थे meeting में,55 लोग आए थे meeting में
9,उसने कहा हाँँँ मैं आऊँगा,उसने कहा हाँँ मैं आऊँगा


**2(f) Importance of Text Normalization**

- Reduces noise (extra spaces, repeated punctuation, inconsistent digits) that otherwise inflates vocabulary size artificially.
- Ensures Unicode-equivalent characters are represented consistently (NFC form), preventing duplicate tokens for the same word.
- Improves accuracy of downstream tasks like tokenization, POS tagging, and text classification.
- Makes case-sensitive Romanized/code-mixed words comparable and searchable.

## Q3. Script Normalization and Transliteration Analysis

In [ ]:
script_samples = [
    "नमस्ते, आप कैसे हैं?",
    "यह एक अच्छा दिन है।",
    "मुझे किताबें पढ़ना पसंद है।",
    "आइए मिलकर काम करें।",
    "आज मौसम बहुत सुहावना है।",
    "আমি বাংলা ভাষায় কথা বলি।",
    "তুমি কেমন আছ?",
    "আজকে আবহাওয়া খুব ভালো।",
    "আমার একটি বই আছে।",
    "আমরা একসাথে কাজ করি।"
]

In [ ]:
def detect_script(text):
    for ch in text:
        if '\u0900' <= ch <= '\u097F':
            return 'Devanagari'
        if '\u0980' <= ch <= '\u09FF':
            return 'Bengali'
    return 'Unknown'

detected_scripts = [detect_script(s) for s in script_samples]
list(zip(script_samples, detected_scripts))

[('नमस्ते, आप कैसे हैं?', 'Devanagari'),
 ('यह एक अच्छा दिन है।', 'Devanagari'),
 ('मुझे किताबें पढ़ना पसंद है।', 'Devanagari'),
 ('आइए मिलकर काम करें।', 'Devanagari'),
 ('आज मौसम बहुत सुहावना है।', 'Devanagari'),
 ('আমি বাংলা ভাষায় কথা বলি।', 'Bengali'),
 ('তুমি কেমন আছ?', 'Bengali'),
 ('আজকে আবহাওয়া খুব ভালো।', 'Bengali'),
 ('আমার একটি বই আছে।', 'Bengali'),
 ('আমরা একসাথে কাজ করি।', 'Bengali')]

In [ ]:
def script_normalize(text):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

normalized_script_samples = [script_normalize(s) for s in script_samples]

In [ ]:
def to_common_form(text, script):
    if script == 'Devanagari':
        return transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    if script == 'Bengali':
        return transliterate(text, sanscript.BENGALI, sanscript.ITRANS)
    return text

transliterated_samples = [to_common_form(t, s) for t, s in zip(normalized_script_samples, detected_scripts)]

In [ ]:
df_script = pd.DataFrame({
    'Original': script_samples,
    'Script': detected_scripts,
    'Normalized': normalized_script_samples,
    'Transliterated (ITRANS)': transliterated_samples
})
df_script

,Original,Script,Normalized,Transliterated (ITRANS)
0,"नमस्ते, आप कैसे हैं?",Devanagari,"नमस्ते, आप कैसे हैं?","namaste, Apa kaise haiM?"
1,यह एक अच्छा दिन है।,Devanagari,यह एक अच्छा दिन है।,yaha eka achChA dina hai|
2,मुझे किताबें पढ़ना पसंद है।,Devanagari,मुझे किताबें पढ़ना पसंद है।,mujhe kitAbeM pa.DhanA pasaMda hai|
3,आइए मिलकर काम करें।,Devanagari,आइए मिलकर काम करें।,Aie milakara kAma kareM|
4,आज मौसम बहुत सुहावना है।,Devanagari,आज मौसम बहुत सुहावना है।,Aja mausama bahuta suhAvanA hai|
5,আমি বাংলা ভাষায় কথা বলি।,Bengali,আমি বাংলা ভাষায় কথা বলি।,Ami vAMlA bhAShAya় kathA vali|
6,তুমি কেমন আছ?,Bengali,তুমি কেমন আছ?,tumi kemana ACha?
7,আজকে আবহাওয়া খুব ভালো।,Bengali,আজকে আবহাওয়া খুব ভালো।,Ajake AvahAoya়A khuva bhAlo|
8,আমার একটি বই আছে।,Bengali,আমার একটি বই আছে।,AmAra ekaTi vai AChe|
9,আমরা একসাথে কাজ করি।,Bengali,আমরা একসাথে কাজ করি।,AmarA ekasAthe kAja kari|


**3(f) Role of Script Normalization in Multilingual and Cross-Lingual NLP**

- Converts text from multiple scripts into a common representation, enabling cross-lingual comparison and search.
- Removes script-specific Unicode inconsistencies (combining marks, nukta variants) that would otherwise be treated as different tokens.
- Essential for building multilingual models, transliteration systems, and cross-lingual information retrieval.
- Helps in reusing NLP tools built primarily for Latin-script/English text on Indic languages via a common transliterated form.

## Q4. Parallel Corpora and Code-Mixed Text Analysis

In [ ]:
parallel_pairs = [
    ("मुझे पानी चाहिए।", "I need water."),
    ("वह स्कूल जा रहा है।", "He is going to school."),
    ("आज मौसम अच्छा है।", "The weather is good today."),
    ("मैं किताब पढ़ रहा हूँ।", "I am reading a book."),
    ("हमें कल मिलना है।", "We have to meet tomorrow."),
    ("यह मेरा घर है।", "This is my house."),
    ("वह बहुत तेज दौड़ता है।", "He runs very fast."),
    ("मुझे संगीत पसंद है।", "I like music."),
    ("बच्चे पार्क में खेल रहे हैं।", "The children are playing in the park."),
    ("यह काम कठिन है।", "This work is difficult."),
    ("वह डॉक्टर बनना चाहती है।", "She wants to become a doctor."),
    ("मैंने खाना बना लिया है।", "I have cooked the food."),
    ("कृपया दरवाजा बंद करें।", "Please close the door."),
    ("यह किताब बहुत रोचक है।", "This book is very interesting."),
    ("हम कल यात्रा पर जाएंगे।", "We will go on a trip tomorrow."),
    ("वह गाना बहुत अच्छा गाती है।", "She sings very well."),
    ("मुझे कॉफी पीना पसंद है।", "I like drinking coffee."),
    ("वे बाजार गए हैं।", "They have gone to the market."),
    ("यह मेरी पसंदीदा जगह है।", "This is my favorite place."),
    ("कल छुट्टी है।", "Tomorrow is a holiday.")
]
len(parallel_pairs)

20

In [ ]:
hindi_sents, english_sents = zip(*parallel_pairs)
hindi_tok = [s.split() for s in hindi_sents]
english_tok = [nltk.word_tokenize(s) for s in english_sents]
aligned = list(zip(hindi_tok, english_tok))
aligned[0]

(['मुझे', 'पानी', 'चाहिए।'], ['I', 'need', 'water', '.'])

In [ ]:
hindi_lengths = [len(t) for t in hindi_tok]
english_lengths = [len(t) for t in english_tok]

hindi_vocab = set(w for s in hindi_tok for w in s)
english_vocab = set(w.lower() for s in english_tok for w in s)

print("Avg Hindi sentence length:", sum(hindi_lengths)/len(hindi_lengths))
print("Avg English sentence length:", sum(english_lengths)/len(english_lengths))
print("Hindi vocabulary size:", len(hindi_vocab))
print("English vocabulary size:", len(english_vocab))

Avg Hindi sentence length: 4.55
Avg English sentence length: 5.75
Hindi vocabulary size: 61
English vocabulary size: 60


In [ ]:
code_mixed_sentences = [
    "Mujhe office jana hai aaj.",
    "Yeh movie bahut interesting hai.",
    "Please जल्दी आओ, हम late ho rahe hain.",
    "Mera phone charge nahi ho raha.",
    "Kal weekend hai, chalo trip पर चलते हैं.",
    "Is project ki deadline कल है.",
    "Mujhe coffee pina hai abhi.",
    "वह meeting में busy है.",
    "Homework complete karke sona.",
    "Traffic bahut zyada hai aaj road par."
]
len(code_mixed_sentences)

10

In [ ]:
def tag_language(token):
    if re.search(r'[\u0900-\u097F]', token):
        return 'HI'
    if re.match(r'^[A-Za-z]+$', token):
        return 'EN'
    return 'OTHER'

def normalize_code_mixed(sentence):
    tokens = sentence.split()
    tagged = [(tok, tag_language(tok)) for tok in tokens]
    normalized = [tok.lower() if tag == 'EN' else tok for tok, tag in tagged]
    return tagged, ' '.join(normalized)

results = [normalize_code_mixed(s) for s in code_mixed_sentences]
for sent, (tagged, norm) in zip(code_mixed_sentences, results):
    print(sent)
    print(tagged)
    print(norm)
    print()

Mujhe office jana hai aaj.
[('Mujhe', 'EN'), ('office', 'EN'), ('jana', 'EN'), ('hai', 'EN'), ('aaj.', 'OTHER')]
mujhe office jana hai aaj.

Yeh movie bahut interesting hai.
[('Yeh', 'EN'), ('movie', 'EN'), ('bahut', 'EN'), ('interesting', 'EN'), ('hai.', 'OTHER')]
yeh movie bahut interesting hai.

Please जल्दी आओ, हम late ho rahe hain.
[('Please', 'EN'), ('जल्दी', 'HI'), ('आओ,', 'HI'), ('हम', 'HI'), ('late', 'EN'), ('ho', 'EN'), ('rahe', 'EN'), ('hain.', 'OTHER')]
please जल्दी आओ, हम late ho rahe hain.

Mera phone charge nahi ho raha.
[('Mera', 'EN'), ('phone', 'EN'), ('charge', 'EN'), ('nahi', 'EN'), ('ho', 'EN'), ('raha.', 'OTHER')]
mera phone charge nahi ho raha.

Kal weekend hai, chalo trip पर चलते हैं.
[('Kal', 'EN'), ('weekend', 'EN'), ('hai,', 'OTHER'), ('chalo', 'EN'), ('trip', 'EN'), ('पर', 'HI'), ('चलते', 'HI'), ('हैं.', 'HI')]
kal weekend hai, chalo trip पर चलते हैं.

Is project ki deadline कल है.
[('Is', 'EN'), ('project', 'EN'), ('ki', 'EN'), ('deadline', 'EN'), ('कल', 'H

**4(f) Challenges of Parallel Data and Code-Mixed Text for NLP Applications**

- Parallel corpora require accurate sentence alignment; manual or noisy alignment introduces errors in downstream MT models.
- Vocabulary and sentence-length mismatches between languages complicate direct statistical comparison.
- Code-mixed text lacks consistent grammar rules, making POS tagging and parsing difficult.
- Language identification at the token level is error-prone for transliterated or ambiguous words.
- Limited annotated code-mixed datasets restrict the performance of standard NLP models on such text.